# Figure 5: HSPA9 case study

Reproduce and audit the manuscript figure from archived results. Model training is documented but is **not executed**.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

HERE = Path.cwd().resolve()
ARCHIVE = HERE.parent if HERE.name == "notebooks" else HERE
if not (ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv").exists():
    raise FileNotFoundError("Run this notebook from the archive root or notebooks directory.")

REGISTRY = pd.read_csv(
    ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv",
    sep="\t",
    keep_default_na=False,
)

default_config = ARCHIVE / "reproducibility" / "configs" / "provided_results.json"
config_path = Path(os.environ.get("SCPLAD_REPRO_CONFIG", default_config)).expanduser().resolve()
REPRO_CONFIG = json.loads(config_path.read_text(encoding="utf-8"))
config_dir = config_path.parent

def resolve_config_path(key, fallback):
    env_key = {
        "source_data_root": "SCPLAD_SOURCE_DATA_ROOT",
        "reproduced_root": "SCPLAD_REPRODUCED_ROOT",
    }[key]
    value = os.environ.get(env_key, REPRO_CONFIG.get("paths", {}).get(key, fallback))
    path = Path(value).expanduser()
    return path if path.is_absolute() else (config_dir / path).resolve()

SOURCE_DATA = resolve_config_path("source_data_root", str(ARCHIVE / "source_data"))
REPRO = resolve_config_path("reproduced_root", str(ARCHIVE / "reproduced"))
REPRO.mkdir(parents=True, exist_ok=True)
os.environ["SCPLAD_REPRO_CONFIG"] = str(config_path)
os.environ["SCPLAD_SOURCE_DATA_ROOT"] = str(SOURCE_DATA)
os.environ["SCPLAD_REPRODUCED_ROOT"] = str(REPRO)
print(f"Figure archive: {ARCHIVE}")
print(f"Input mode: {REPRO_CONFIG['mode']}")
print(f"Source data: {SOURCE_DATA}")
print(f"Output root: {REPRO}")

## Data preparation and result provenance

The HSPA9 case study uses saved external causal-target effects,
top-response-gene correlation matrices, and q10-q90 expression ranges.
It is an analysis of stored predictions from `XCL-MAIN` and the STATE
and TxPert comparators; no model training is run here.

In [ ]:
panel_map = REGISTRY.loc[REGISTRY["figure"].eq("Fig5")].copy()
required = ["source_data", "training_code", "evaluation_code", "plot_code", "canonical_panel"]
display(panel_map[["panel", "panel_type", "experiment_id", "claim_or_role", "status"]])

def archived_paths_exist(value, base):
    if value == "NA":
        return True
    return all((base / item).exists() for item in str(value).split(";"))

for column in ["source_data", "plot_code", "canonical_panel"]:
    missing = [
        value for value in panel_map[column]
        if not archived_paths_exist(value, ARCHIVE)
    ]
    assert not missing, f"Missing {column}: {missing}"
print("Panel-level figure inputs and plotting assets are present.")

In [ ]:
source = SOURCE_DATA / "Fig5"
targets = pd.read_csv(source / "hspa9_external_causal_target_recovery_source_data.csv")
ranges = pd.read_csv(source / "hspa9_q10_q90_range_data.csv")
display(targets.groupby("model").size().rename("external_targets"))
display(ranges.groupby("model").size().rename("display_genes"))

In [ ]:
for script in [
    "make_hspa9_external_panels_20260721.py",
    "make_hspa9_csa_heatmap_20260721.py",
    "make_hspa9_range_fig5_base_20260727.py",
]:
    subprocess.run(
        [sys.executable, str(ARCHIVE / "scripts" / "Fig5" / script)],
        check=True,
    )

In [ ]:
outputs = [
    ("a", "External causal-target recovery", REPRO / "Fig5" / "figure5a_hspa9_external_targets.svg"),
    ("b", "Response-gene correlation structure", REPRO / "Fig5" / "figure5b_hspa9_csa_heatmap.svg"),
    ("c", "Expression-range comparison", REPRO / "Fig5" / "figure5c_hspa9_expression_range.svg"),
]
assert all(path.exists() for _, _, path in outputs)
for panel, title, path in outputs:
    display(Markdown(f"### Fig. 5{panel}: {title}"))
    display(SVG(filename=str(path)))